# PPO best response against a frozen neural policy

This notebook trains role-separated stochastic PPO responders against the same Deep CFR reference target. Episodes provide complete terminal returns; the value network is only a variance-reduction baseline. Exact fixed-policy evaluation measures the resulting responder.

In [ ]:
from pathlib import Path
import gc
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.serialization import load_policy
from liars_poker.policies.neural import NeuralPolicy, compile_neural_to_dense
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.br_ppo import PPOBRTrainer
from liars_poker.eval.match_dense import evaluate_dense_response

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
spec = GameSpec(
    ranks=4, suits=4, hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)
reference_root = REPO_ROOT / 'artifacts' / 'deep_cfr_reference_runs'
candidates = sorted(
    (p for p in reference_root.glob(f'{spec.to_short_str()}___*')
     if (p / 'policy' / 'metadata.json').exists()),
    key=lambda p: p.stat().st_mtime, reverse=True,
)
if not candidates:
    raise FileNotFoundError('No completed reference policy found.')
REFERENCE_RUN_DIR = candidates[0]
target, target_spec = load_policy(str(REFERENCE_RUN_DIR / 'policy'))
assert target_spec == spec and isinstance(target, NeuralPolicy)
target.model_p1.to(device); target.model_p2.to(device)
target.device = torch.device(device); target.eval()

target_dense = compile_neural_to_dense(target, batch_size=65_536)
_, meta = best_response_dense(spec, target_dense, store_state_values=False)
exact_first, exact_second = meta['computer'].exploitability()
exact_exploitability = exact_first + exact_second - 1.0
print('reference:', REFERENCE_RUN_DIR)
print('exact exploitability:', exact_exploitability)

In [ ]:
SEEDS = (7, 17, 27)
BUDGETS_S = (0, 15, 30, 60, 120, 240, 480, 900)
TRAINER_KWARGS = dict(
    hidden_sizes=(512, 512),
    value_hidden_sizes=(256, 256),
    device=device,
    learning_rate=3e-4,
    value_learning_rate=1e-3,
    ppo_epochs=4,
    minibatch_size=4096,
    clip_ratio=0.2,
    entropy_coef=0.01,
    value_coef=0.5,
)
EPISODES_PER_ROLE = 8192
ROLLOUT_BATCH_SIZE = 8192

def exact_value(trainer):
    responder = compile_neural_to_dense(trainer.policy(), batch_size=65_536)
    p_first, p_second = evaluate_dense_response(spec, target_dense, responder)
    return p_first, p_second, p_first + p_second - 1.0

frames = []
for seed in SEEDS:
    print(f'\n=== seed={seed} ===')
    trainer = PPOBRTrainer(spec, target, seed=seed, **TRAINER_KWARGS)
    measured_s = 0.0
    budget_idx = 0
    best_first = -np.inf
    best_second = -np.inf
    rows = []
    last = None
    while budget_idx < len(BUDGETS_S):
        if measured_s >= BUDGETS_S[budget_idx]:
            p_first, p_second, current = exact_value(trainer)
            best_first = max(best_first, p_first)
            best_second = max(best_second, p_second)
            best = best_first + best_second - 1.0
            row = {
                'seed': seed,
                'requested_budget_s': BUDGETS_S[budget_idx],
                'measured_training_s': measured_s,
                'iteration': trainer.iteration,
                'current exploitability': current,
                'best discovered exploitability': best,
                'oracle gap': exact_exploitability - best,
            }
            if last is not None:
                row.update({
                    'role 1 policy loss': last['roles'][0]['policy_loss'],
                    'role 2 policy loss': last['roles'][1]['policy_loss'],
                    'role 1 value loss': last['roles'][0]['value_loss'],
                    'role 2 value loss': last['roles'][1]['value_loss'],
                    'role 1 entropy': last['roles'][0]['entropy'],
                    'role 2 entropy': last['roles'][1]['entropy'],
                    'collect s': sum(r['collect_s'] for r in last['roles']),
                    'fit s': sum(r['fit_s'] for r in last['roles']),
                })
            rows.append(row)
            print(f"train={measured_s:7.1f}s iter={trainer.iteration:4d} best={best:.6f} gap={exact_exploitability-best:.6f}")
            budget_idx += 1
            continue
        start = time.perf_counter()
        last = trainer.run_iteration(
            episodes_per_role=EPISODES_PER_ROLE,
            rollout_batch_size=ROLLOUT_BATCH_SIZE,
        )
        measured_s += time.perf_counter() - start
    frames.append(pd.DataFrame(rows))
    del trainer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

results = pd.concat(frames, ignore_index=True)
results.tail()

In [ ]:
dqn_results = None
dqn_root = REPO_ROOT / 'artifacts' / 'neural_br_reference_calibration'
dqn_candidates = sorted(dqn_root.glob('*/results.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if dqn_candidates:
    dqn_results = pd.read_csv(dqn_candidates[0])
    print('DQN comparison:', dqn_candidates[0])

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for seed, frame in results.groupby('seed'):
    positive = frame['measured_training_s'] > 0
    x = frame.loc[positive, 'measured_training_s']
    axes[0].plot(x, frame.loc[positive, 'best discovered exploitability'], marker='o', label=f'PPO seed {seed}')
    axes[1].loglog(x, frame.loc[positive, 'oracle gap'].clip(lower=1e-10), marker='o', label=f'PPO seed {seed}')
    axes[2].plot(x, frame.loc[positive, 'role 1 entropy'], marker='o', label=f'P1 seed {seed}')
    axes[2].plot(x, frame.loc[positive, 'role 2 entropy'], marker='x', linestyle='--', label=f'P2 seed {seed}')
if dqn_results is not None:
    dqn_mean = dqn_results.groupby('requested_budget_s').agg(
        measured=('measured_training_s', 'mean'),
        best=('best_exact_discovered', 'mean'),
        gap=('exact_oracle_gap', 'mean'),
    )
    positive = dqn_mean['measured'] > 0
    axes[0].plot(dqn_mean.loc[positive, 'measured'], dqn_mean.loc[positive, 'best'], color='black', linewidth=2.5, label='DQN mean')
    axes[1].loglog(dqn_mean.loc[positive, 'measured'], dqn_mean.loc[positive, 'gap'].clip(lower=1e-10), color='black', linewidth=2.5, label='DQN mean')
axes[0].axhline(exact_exploitability, color='grey', linestyle='--', label='exact ceiling')
axes[0].set_xscale('log')
axes[0].set_title('Best discovered exploitability')
axes[1].set_title('Exact oracle gap')
axes[2].set_xscale('log')
axes[2].set_title('Policy entropy')
for ax in axes:
    ax.set_xlabel('Measured training seconds')
    ax.grid(alpha=0.3, which='both')
    ax.legend(fontsize=8)
plt.tight_layout()

results.groupby('seed').tail(1).set_index('seed')